# BurstGPT 3: baseline capacity and deadline diagnostics

**Purpose:** find out when FCFS, length-aware, and static SLO-aware scheduling struggle, before spending time retraining DQN.

Run the cells in order in a fresh Colab **CPU runtime**. Starting from the raw CSV, this notebook prepares data, selects two short development windows, compares three policies at 1×/2×/4×/8× load, plots the results, and exports a self-contained team report. There is **no RL training**, checkpoint requirement, or replay of the long held-out test groups. This is a capacity study that informs the *next* RL experiment.

A simulation models GPU behavior but executes on CPU. Switching Colab to a GPU will not accelerate this sweep. The default is 24 short simulations, with each completed case cached independently. The first case prints its elapsed time; use that to gauge cost rather than assuming a fixed runtime.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/xfgao210/llm-request-triage.git"
BRANCH = "burstgpt_3"  # must contain this notebook AND experiments/baseline_capacity_sweep.py
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
REPO = Path.cwd() if Path("experiments/baseline_capacity_sweep.py").is_file() else Path("/content/llm-request-triage")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
assert (REPO / "experiments/baseline_capacity_sweep.py").is_file(), (
    "Checkout lacks the capacity sweep. Use the branch/commit containing this notebook; "
    "do not reuse an older Colab checkout."
)
os.chdir(REPO)
# Keep an existing checkout unchanged: never silently pull a different simulator mid-experiment.
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[analysis]", "pandas>=2"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Checkout:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 1. Settings and source CSV

Upload the raw `BurstGPT_3.csv` to Drive once (no results ZIP or trained model is needed). Update `CSV_PATH` below. Alternatively, disable Drive and upload the CSV using Colab's Files panel.

Store outputs on Drive to preserve completed cases across runtime disconnects. Keep the same `RUN_NAME` to resume identical settings; **use a new name if you change code, selected windows, SLO, capacity, or load multipliers**. Never overwrite a previous experiment to hide an unfavorable result.

The default hardware comes from your existing BurstGPT 3 experiment config. Its GPU name is a label; actual timing comes from the recorded service profile.


In [ ]:
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

CSV_PATH = Path("/content/drive/MyDrive/BurstGPT_3.csv")
OUTPUT_ROOT = Path("/content/drive/MyDrive/burstgpt3_capacity") if IN_COLAB and USE_DRIVE else REPO / "results/burstgpt3_capacity"
RUN_NAME = "development_v1"
DATA_DIR = REPO / "data/burstgpt3"
ANALYSIS_DIR = REPO / "results/burstgpt3_capacity_analysis"
RANGE_CONFIG = REPO / "configs/burstgpt3_ranges.json"

DEVELOPMENT_SPLIT = "validation"  # "train" or "validation"; NEVER "test"
WINDOW_SECONDS = 900             # two continuous 15-minute development blocks
LOAD_MULTIPLIERS = [1, 2, 4, 8]  # divide arrival gaps by this factor; do NOT scale the SLO
SLO_SECONDS = 5.0
TIMELINE_BIN_SECONDS = 10.0      # in compressed simulation time
MAX_REQUESTS_PER_WINDOW = 30000  # safety guard; shorten windows if exceeded

# Optional explicit windows instead of traffic-only automatic selection.
# Both must lie within the configured train/validation ranges; no overlapping windows.
# MANUAL_WINDOWS = [
#     {"name": "typical_development", "split": "validation",
#      "start_timestamp_s": ..., "end_timestamp_s": ...},
#     {"name": "burst_development", "split": "validation",
#      "start_timestamp_s": ..., "end_timestamp_s": ...},
# ]
MANUAL_WINDOWS = None


In [ ]:
import json
from IPython.display import display, FileLink
import pandas as pd

from scripts.analyze_burstgpt_traffic import analyze
from llm_triage.workload.time_ranges import prepare_ranges
from llm_triage.workload.burstgpt import sha256
from llm_triage.simulation import HardwareConfig, ServiceProfile
from experiments.baseline_capacity_sweep import (
    select_development_windows, validate_windows, load_selected_rows,
    run_sweep, write_json,
)

assert CSV_PATH.is_file(), f"Upload BurstGPT_3.csv and set CSV_PATH. Missing: {CSV_PATH}"
range_config = json.loads(RANGE_CONFIG.read_text())
assert sha256(CSV_PATH) == range_config["source_sha256"], "CSV differs from the pinned BurstGPT 3 source."
manifest_path = DATA_DIR / "processed/manifest.json"
if not manifest_path.exists():
    if not (ANALYSIS_DIR / "summary.json").exists():
        analyze(CSV_PATH, ANALYSIS_DIR)
    analysis_summary = json.loads((ANALYSIS_DIR / "summary.json").read_text())
    assert analysis_summary["source"]["sha256"] == range_config["source_sha256"], "Use a fresh ANALYSIS_DIR."
    prepare_ranges(CSV_PATH, ANALYSIS_DIR, DATA_DIR, RANGE_CONFIG)
manifest = json.loads(manifest_path.read_text())
assert manifest["range_config"] == range_config, "Prepared ranges differ; choose a fresh DATA_DIR."
assert manifest["source_sha256"] == range_config["source_sha256"]
assert sha256(DATA_DIR / "processed/burstgpt_windows.csv") == manifest["selected_sha256"]
# Full preparation includes held-out data for compatibility, but the sweep only loads
# selected train/validation requests. Analyzer's automatic 60/20/20 labels are NOT used.
print("Data verified. Baseline output-length estimate (training only):",
      manifest["estimated_output_tokens_train_mean"])


## 2. Choose and inspect development windows before running policies

Automatic selection considers disjoint, complete 15-minute blocks inside the **explicit validation range**. It chooses the block with highest 5-minute request rate as “burst,” then a block nearest the median request count among the remaining nonempty candidates as “typical.” Selection uses traffic only, never policy scores. These labels are relative to the development data, not guarantees of global representativeness.

All requests and idle gaps inside each window are retained. Each window starts with an empty queue, but **there are no internal 5-minute resets**. This is a cold-start capacity probe, not an exact continuation of the original long replay; inspect whether a burst hits the boundary. Manual windows can include more lead-in/recovery time. No-record periods could reflect inactivity or missing collection.

Review the timestamps and plots before running. Timestamps are source-relative seconds, suitable for the JSON config—not calendar dates. Held-out mixed/peak test groups remain reserved for final evaluation.


In [ ]:
from dataclasses import asdict
import matplotlib.pyplot as plt
from io import BytesIO
import numpy as np

windows = MANUAL_WINDOWS if MANUAL_WINDOWS is not None else select_development_windows(
    manifest, split=DEVELOPMENT_SPLIT, seconds=WINDOW_SECONDS)
validate_windows(manifest, windows)
selected_rows = load_selected_rows(DATA_DIR / "processed/burstgpt_windows.csv", windows)
base_config = json.loads((REPO / "configs/burstgpt3_initial.json").read_text())
hardware = HardwareConfig(**base_config["hardware"])
profile = ServiceProfile()
RUN_DIR = OUTPUT_ROOT / RUN_NAME
FIGURE_DIR = RUN_DIR / "figures"
display(pd.DataFrame(windows))
display(pd.DataFrame([{"slo_s": SLO_SECONDS, **asdict(hardware)}]))
print("Service profile:", asdict(profile))
print("Expected cases:", len(windows) * len(LOAD_MULTIPLIERS) * 3)

fig, axes = plt.subplots(len(windows), 1, figsize=(13, 3 * len(windows)), squeeze=False)
for ax, window in zip(axes[:, 0], windows):
    rows = selected_rows[window["name"]]
    start, end = window["start_timestamp_s"], window["end_timestamp_s"]
    bins = np.linspace(start, end, max(2, int((end-start) / 30) + 1))
    counts, edges = np.histogram([r.timestamp_s for r in rows], bins=bins)
    ax.stairs(counts / np.diff(edges), edges)
    ax.set(title=window["name"], xlabel="Source timestamp (seconds)", ylabel="Arrivals / second")
    ax.ticklabel_format(axis="x", style="plain", useOffset=False)
    ax.grid(alpha=.2)
fig.tight_layout()
traffic_preview = BytesIO()
fig.savefig(traffic_preview, format="png", dpi=150)
plt.show()


## 3. Run the paired baseline sweep

Every policy receives the same request IDs, input/output lengths, and arrival order within a case. At load multiplier *m*, the whole arrival window and all its idle gaps are compressed by *m*. Deadlines remain **arrival + 5 seconds** by default. Hardware, timing profile, and training-only output-length estimate stay fixed.

FCFS and length-aware do not proactively drop; static SLO-aware can. Simulator resource-infeasibility handling is shared by all policies. Each replay drains fully after arrivals stop; overdue requests are not automatically expired just to improve scores. Drops count against SLO attainment.

Completed cases are reused only if inputs, code, and cached file hashes still match. An interrupted case restarts; completed cases do not. Keep this runtime open while the sweep runs.


In [ ]:
summaries = run_sweep(
    DATA_DIR, RUN_DIR, windows, multipliers=LOAD_MULTIPLIERS, slo_s=SLO_SECONDS,
    hardware=hardware, profile=profile, timeline_bin_s=TIMELINE_BIN_SECONDS,
    max_requests=MAX_REQUESTS_PER_WINDOW,
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
(FIGURE_DIR / "selected_traffic.png").write_bytes(traffic_preview.getvalue())
summary = pd.read_csv(RUN_DIR / "summary.csv")
display(summary[[
    "window", "multiplier", "policy", "arrivals", "slo_attainment", "drop_rate",
    "window_goodput_rps", "p95_queue_wait_s", "p95_completion_latency_s",
    "busy_fraction_in_window", "isolated_exceeds_slo_fraction",
    "missed_but_isolated_within_slo", "wall_seconds",
]])
print("Total measured simulation/diagnostic wall time (cached cases retain original times):",
      round(summary.wall_seconds.sum(), 1), "seconds")


## 4. Compare capacity curves and diagnose deadline misses

- **SLO attainment:** on-time completions / all arrivals, including dropped requests.
- **Window goodput:** on-time completions *inside the arrival window* / compressed window duration. Completions during draining still count toward cohort SLO attainment, but not this window rate.
- **Queue wait / completion latency:** p95 among requests with observed service start / completion. Drops have no completion latency; always read these alongside drop rate.
- **Busy / batch / KV utilization:** duration-weighted iteration snapshots, clipped to the arrival window. Batch/KV are approximate diagnostics, not real GPU counters. “Busy” alone does not mean all batch slots are full.
- **Wasted GPU fraction:** the model's allocated GPU time spent on requests that fail their SLO, divided by busy GPU time over the **entire drained replay**.
- **Isolated exceeds SLO:** actual output length used offline to compute batch=1 service time. This flags requests that are already too slow alone under the current model. It is **not a policy input**. Misses in the other category may involve queueing, contention, memory limits, or scheduling; they are not proven avoidable.

If FCFS stays strong even at higher loads, report that honestly. If attainment falls while queue wait rises, that is evidence of a queueing failure mode. If isolated service already exceeds SLO, scheduling alone cannot fix those requests under this model.


In [ ]:
from matplotlib.ticker import PercentFormatter

plot_metrics = [
    ("slo_attainment", "SLO attainment", True),
    ("window_goodput_rps", "Window on-time goodput (req/s)", False),
    ("p95_queue_wait_s", "p95 queue wait (served, seconds)", False),
    ("p95_completion_latency_s", "p95 completion latency (seconds)", False),
    ("drop_rate", "Drop rate", True),
    ("full_replay_wasted_gpu_fraction", "Wasted GPU fraction (incl. drain)", True),
    ("busy_fraction_in_window", "Busy fraction (arrival window)", True),
    ("batch_slot_utilization_in_window", "Batch-slot utilization (approx.)", True),
    ("kv_utilization_in_window", "KV utilization (approx.)", True),
]
for window in windows:
    frame = summary[summary.window == window["name"]]
    fig, axes = plt.subplots(3, 3, figsize=(15, 11))
    for ax, (metric, title, fraction) in zip(axes.flat, plot_metrics):
        for policy, values in frame.groupby("policy"):
            values = values.sort_values("multiplier")
            ax.plot(values.multiplier, values[metric], "o-", label=policy)
        ax.set(title=title, xlabel="Arrival multiplier")
        ax.set_xticks(sorted(frame.multiplier.unique()))
        if fraction:
            ax.yaxis.set_major_formatter(PercentFormatter(1))
            lower, upper = ax.get_ylim()
            ax.set_ylim(max(0, lower), min(1, upper))
        ax.grid(alpha=.2)
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(window["name"])
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / (window["name"] + "_curves.png"), dpi=150)
    plt.show()

diagnostic_columns = ["window", "multiplier", "policy", "arrivals", "slo_met",
                      "isolated_exceeds_slo", "missed_but_isolated_within_slo",
                      "max_queue_in_window", "preemptions", "drain_s"]
display(summary[diagnostic_columns])


In [ ]:
# Inspect queue evolution for native and highest offered load, in source-time coordinates.
inspect_loads = sorted({min(LOAD_MULTIPLIERS), max(LOAD_MULTIPLIERS)})
fig, axes = plt.subplots(len(windows), len(inspect_loads),
                         figsize=(7 * len(inspect_loads), 3.5 * len(windows)), squeeze=False)
for row_index, window in enumerate(windows):
    for col_index, multiplier in enumerate(inspect_loads):
        ax = axes[row_index, col_index]
        for policy in summary.policy.unique():
            path = RUN_DIR / window["name"] / f"load_{multiplier:g}" / policy / "timeline.csv"
            timeline = pd.read_csv(path)
            ax.plot(timeline.source_start_s, timeline.mean_queue, label=policy)
        ax.set(title=f'{window["name"]}, {multiplier:g}x',
               xlabel="Source timestamp (seconds; time compressed in simulation)", ylabel="Mean queue length")
        ax.ticklabel_format(axis="x", style="plain", useOffset=False)
        ax.grid(alpha=.2)
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "queue_timelines.png", dpi=150)
plt.show()


## 5. Share results and decide the next experiment

Use these development results to predeclare an underloaded, near-saturation, and overloaded setting (if the sweep actually reaches them). Compare all policies, not just FCFS. Do not pick capacity/SLO purely because it makes RL win.

Next, fix the inconsistent `episode_progress` representation, retrain at the chosen settings, and evaluate frozen and causal-adapting DQN plus these same baselines under identical held-out traffic/protocols. That is a separate experiment; this notebook does not change observations, rewards, or model weights.

**Model limitations:** the timing defaults come from a batch=1 RTX 5070/Qwen calibration, with an assumed, unvalidated batch-contention term. The hardware name does not make it a real GPU benchmark. Cold starts and only two short windows limit generalization. Repeat with additional predeclared development windows and separate capacity/profile sensitivity runs before claiming real-server results.


In [ ]:
import base64
import html
import zipfile

assert json.loads((RUN_DIR / "sweep.json").read_text())["status"] == "completed"
write_json(RUN_DIR / "selected_windows.json", windows)
limitations = (
    "Development-only synthetic capacity sweep; not an RL comparison or real GPU benchmark. "
    "Each window starts empty and drains fully. SLO includes drops in its denominator. "
    "Window goodput excludes post-window completions; wasted GPU time includes drain. "
    "Latency quantiles exclude requests lacking an observed latency. "
    "Batch/KV utilization uses approximate iteration snapshots. "
    "Batch contention is assumed, not calibrated. True output lengths are used only in offline diagnostics. "
    "Missed-but-isolated-within-SLO does not mean proven avoidable. "
    "Window labels reflect development-relative traffic, not guaranteed global regimes."
)
sections = [
    "<!doctype html><html><head><meta charset='utf-8'><title>Baseline capacity sweep</title>",
    "<style>body{font:15px system-ui;margin:30px;max-width:1500px}table{display:block;overflow-x:auto;font-size:12px;border-collapse:collapse}"
    "td,th{padding:5px;border:1px solid #ddd}img{max-width:100%}pre{white-space:pre-wrap}</style></head><body>",
    "<h1>BurstGPT 3 baseline capacity study</h1><p>" + html.escape(limitations) + "</p>",
    "<h2>Selected continuous development windows</h2>", pd.DataFrame(windows).to_html(index=False),
    "<h2>Paired policy results</h2>", summary.to_html(index=False),
    "<h2>Figures</h2>",
]
for path in sorted(FIGURE_DIR.glob("*.png")):
    encoded = base64.b64encode(path.read_bytes()).decode()
    sections.append(f"<h3>{html.escape(path.stem)}</h3><img src='data:image/png;base64,{encoded}'>")
sections += ["<h2>Reproduction settings and code hashes</h2><pre>",
             html.escape((RUN_DIR / "sweep.json").read_text()), "</pre></body></html>"]
report_path = RUN_DIR / "capacity_report.html"
report_path.write_text("\n".join(sections))
archive_path = OUTPUT_ROOT / (RUN_NAME + "_team_report.zip")
# Explicit output allowlist: no raw CSV, prepared dataset, secrets, or model weights.
files = [RUN_DIR / "summary.csv", RUN_DIR / "sweep.json",
         RUN_DIR / "selected_windows.json", report_path]
files += list(FIGURE_DIR.glob("*.png"))
for window in windows:
    for multiplier in LOAD_MULTIPLIERS:
        for policy in summary.policy.unique():
            folder = RUN_DIR / window["name"] / f"load_{multiplier:g}" / policy
            files += [folder / name for name in ("summary.json", "case.json", "timeline.csv", "request_diagnostics.csv")]
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(RUN_DIR))
print("Report:", report_path)
print("Shareable ZIP:", archive_path)
display(FileLink(str(report_path)))
display(FileLink(str(archive_path)))
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(archive_path))
